# Part B — FreshCart ELT on Snowflake

Run every cell top to bottom. Each section is labelled to match the assignment
(B1–B5). Screenshot the output of each cell as you go for the report appendix.

**Note:** the mock dataset in `freshcart_lib.py` was built without the original
course notebook (not available when this was put together), so the dollar
figures here are illustrative, not the instructor's exact numbers — but the
*shape* of the pipeline (8 raw rows, 6 clean, 2 quarantined, 4 after dedup,
SCD2 tier changes, idempotent MERGE) matches what the assignment asks you to
trace. Copy the numbers *your* run produces into the report template.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))

import freshcart_lib as fc

conn = fc.get_connection()
fc.run_ddl(conn)
print("Connected and DDL applied.")

## B1 — Run the baseline

In [ ]:
fc.run_baseline_load(conn)
fc.run_transform(conn, include_currency_check=False)

bronze_count = fc.query_df(conn, f"SELECT COUNT(*) AS n FROM {fc.DATABASE}.BRONZE.ORDERS_RAW")
checked = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_CHECKED ORDER BY order_id, source")
clean_count = (checked["REASONS"].apply(len) == 0).sum() if len(checked) else 0
quarantine = checked[checked["REASONS"].apply(len) > 0]
dedup = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_DEDUP ORDER BY order_id")
gold = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.GOLD.FCT_ORDERS ORDER BY order_id")

print("Raw rows in bronze:", bronze_count.iloc[0, 0])
print("Rows that pass cleaning:", clean_count)
print("Rows in quarantine:", len(quarantine))
print("Orders after deduplication:", len(dedup))
print("Gold rows:", len(gold), "| revenue:", gold["AMOUNT_INR"].sum(), "| loyalty points:", gold["LOYALTY_POINTS"].sum())
print("Share of rows rejected: {:.1%}".format(len(quarantine) / bronze_count.iloc[0, 0]))
quarantine[["order_id", "source", "REASONS"]] if len(quarantine) else quarantine

In [ ]:
checked  # inspect silver.orders_checked

In [ ]:
dedup  # inspect silver.orders_dedup

In [ ]:
gold  # inspect gold.fct_orders

## B2 — Trace one record by hand (ORD-1003, then ORD-1005 vs ORD-1001)

The queries below pull exactly what you need to write the trace: the two raw
bronze rows, how `SQL_10_ORDERS_CLEAN` maps each source's different field
names onto the same columns, which copy the dedup `QUALIFY ROW_NUMBER()` line
keeps, and the gold-table arithmetic.

In [ ]:
fc.query_df(conn, f"""
    SELECT source, raw_json FROM {fc.DATABASE}.BRONZE.ORDERS_RAW
    WHERE raw_json:order_id::STRING = 'ORD-1003' OR raw_json:cust_id::STRING IS NOT NULL AND raw_json:order_id::STRING = 'ORD-1003'
""")

In [ ]:
fc.query_df(conn, f"""
    SELECT * FROM {fc.DATABASE}.SILVER.ORDERS_CHECKED WHERE order_id = 'ORD-1003' ORDER BY updated_at
""")
# Whichever row has the latest updated_at is the one QUALIFY ROW_NUMBER() ... = 1 keeps.

In [ ]:
fc.query_df(conn, f"""
    SELECT g.*, s.effective_from, s.effective_to
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g
    JOIN {fc.DATABASE}.DIM.CUSTOMERS_SCD2 s ON s.customer_id = g.customer_id AND s.tier = g.tier_used
    WHERE g.order_id IN ('ORD-1001', 'ORD-1005', 'ORD-1003')
    ORDER BY g.order_id
""")
# amount_inr * multiplier / 100, floored, is loyalty_points -- the SCD2 join (dim.customers_scd2,
# matched on order_date BETWEEN effective_from AND effective_to) is why ORD-1001 prices at
# Priya's Silver tier and ORD-1005 at her later Gold tier.

## B3 — Add a validation rule (unsupported_currency)

`include_currency_check=False` above is the "before" state. Re-running with
`include_currency_check=True` adds the line inside `ARRAY_CONSTRUCT_COMPACT(...)`
in `freshcart_lib.sql_10_orders_clean()` — open that function and you'll see
exactly the one line to add if you were doing this by hand in the SQL editor.

In [ ]:
before = fc.get_reasons_for_order(conn, "ORD-1006")
print("Reasons for ORD-1006 BEFORE:", before)

fc.run_transform(conn, include_currency_check=True)

after = fc.get_reasons_for_order(conn, "ORD-1006")
print("Reasons for ORD-1006 AFTER: ", after)

gold_after = fc.query_df(conn, f"SELECT SUM(amount_inr) AS revenue, COUNT(*) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS")
print("Gold total after adding the check:", gold_after)
# ORD-1006 was already in quarantine (negative_amount) before this change, so it does NOT add
# a second quarantine row -- ARRAY_CONSTRUCT_COMPACT just appends a second reason to the SAME
# row's `reasons` array. The gold total is unchanged because ORD-1006 was excluded from
# silver.orders_dedup (and therefore from gold) before AND after -- WHERE ARRAY_SIZE(reasons) = 0
# in SQL_20_ORDERS_DEDUP already filtered it out.

## B4 — Change history and idempotency (SCD2)

In [ ]:
before_b4 = fc.query_df(conn, f"""
    SELECT g.order_id, g.tier_used, g.loyalty_points, g.published_at
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g WHERE g.order_id = 'ORD-1002'
""")
total_points_before = fc.query_df(conn, f"SELECT SUM(loyalty_points) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS").iloc[0, 0]
print("Before B4 -- ORD-1002:", before_b4.to_dict("records"), "| total loyalty points:", total_points_before)

# PREDICTION: Arjun's new Gold tier is effective_from 2026-08-15, which is AFTER
# ORD-1002's order_date (2026-07-02) -- so the point-in-time join should NOT pick up
# Gold for ORD-1002. Write your own prediction in the report BEFORE running this cell.

fc.add_tier_event_and_rerun(conn, "CUST-02", "Gold", "2026-08-15")

scd2_rows = fc.query_df(conn, f"SELECT * FROM {fc.DATABASE}.DIM.CUSTOMERS_SCD2 WHERE customer_id = 'CUST-02'")
after_b4 = fc.query_df(conn, f"""
    SELECT g.order_id, g.tier_used, g.loyalty_points, g.published_at
    FROM {fc.DATABASE}.GOLD.FCT_ORDERS g WHERE g.order_id = 'ORD-1002'
""")
total_points_after = fc.query_df(conn, f"SELECT SUM(loyalty_points) AS n FROM {fc.DATABASE}.GOLD.FCT_ORDERS").iloc[0, 0]

print("CUST-02 SCD2 rows:\n", scd2_rows)
print("After B4 -- ORD-1002:", after_b4.to_dict("records"), "| total loyalty points:", total_points_after)

In [ ]:
bronze_count_after = fc.query_df(conn, f"SELECT COUNT(*) AS n FROM {fc.DATABASE}.BRONZE.ORDERS_RAW")
print("Bronze row count after this re-run:", bronze_count_after.iloc[0, 0])
# Bronze grows by 8 every run because load_file() uses COPY INTO ... FORCE=TRUE on the SAME
# local files -- bronze is an append-only raw landing layer by design, with no dedup logic at
# all. row_hash (order_id + customer_id + amount_inr + tier) + the gold MERGE's
# "ON t.row_hash = src.row_hash" is where idempotency actually lives: if an order's business
# columns and tier haven't changed between runs, its row_hash is identical, so MERGE just
# updates published_at (or does nothing) instead of inserting a duplicate gold row. Only rows
# whose row_hash actually changed (like ORD-1002 if its tier changed) get a new published_at.

## B5 — Add a data-quality check

In [ ]:
checks = fc.quality_checks(conn)
print("quality_checks() --", checks)
assert all(v == 0 for v in checks.values()), "A check that should be 0 is failing!"

In [ ]:
# Make the NEW check (future_dated_orders) fail on purpose: insert a mock order dated
# in the future directly into silver.orders_dedup, show PROBLEM, then remove it again.
conn.cursor().execute(f"""
    INSERT INTO {fc.DATABASE}.SILVER.ORDERS_DEDUP
        (order_id, customer_id, amount, currency, order_date, updated_at, source, reasons)
    SELECT 'ORD-9999', 'CUST-01', 999.00, 'INR', DATEADD(day, 30, CURRENT_DATE()), CURRENT_TIMESTAMP(), 'web', ARRAY_CONSTRUCT()
""")

checks_with_problem = fc.quality_checks(conn)
print("PROBLEM ->", checks_with_problem)
assert checks_with_problem["future_dated_orders"] > 0

conn.cursor().execute(f"DELETE FROM {fc.DATABASE}.SILVER.ORDERS_DEDUP WHERE order_id = 'ORD-9999'")
checks_clean_again = fc.quality_checks(conn)
print("back to clean ->", checks_clean_again)

**Why ELT, not ETL, for FreshCart?** Snowflake's compute is cheap and
elastic, so it's more efficient to load the raw JSON as-is (schema-on-read)
and let SQL do every transform step — bronze/silver/quarantine/gold — inside
the warehouse, rather than paying for a separate transform server before
loading. It also means the original raw payloads are always available if a
transform rule turns out to be wrong (just re-run the SQL, you don't need to
re-extract).

**Where would ETL still win?** When the source system can't be trusted to
hold raw data securely or cheaply — e.g. a legal/compliance requirement to
strip or mask PII *before* it ever lands in a shared warehouse (so no raw,
unmasked customer data is ever stored even transiently), ETL's
transform-before-load model is the safer choice.